# Ports & Adapters: interactive walkthrough

One **core** (a `Task` and the `CompleteTask` use case) and two different sets of **adapters** plugged into it.
The core states what it needs as **ports** (`ITaskRepository`, `INotifier`), and adapters fill those ports with a technology.

This notebook replays the script's two composition roots, `demo_production()` and `demo_test()`, one step per cell.
It must live next to `ports_adapters_toy.py` (it imports it by name). Launch with:

```bash
uv run --with jupyterlab jupyter lab ports_adapters.ipynb
```

In [1]:
import json
import tempfile
from pathlib import Path

from ports_adapters_toy import (
    CompleteTask,                                     # core: the use case (driving port = .execute)
    Task,                                             # core: the entity and its rule
    InMemoryTaskRepository, JsonFileTaskRepository,   # driven adapters for ITaskRepository
    ConsoleNotifier, SpyNotifier,                     # driven adapters for INotifier
    cli,                                              # driving adapter
)

## 1. Production wiring: JSON file + console

This is the body of `demo_production()`. The adapters are real technology: a file on disk and stdout.

### Seed the store

The file is the outside world. It holds plain JSON, with no `Task` objects in it.

In [2]:
path = Path(tempfile.mkdtemp()) / "tasks.json"
path.write_text(json.dumps({"1": {"id": 1, "title": "Write report", "done": False}}))
print(path.read_text())

{"1": {"id": 1, "title": "Write report", "done": false}}


### Wire the core to the adapters

This one line is the **composition root**: the only place that knows both the core and the technology.
`CompleteTask` sees only an `ITaskRepository` and an `INotifier`. It can't tell that a JSON file and a console are behind them.

In [3]:
use_case = CompleteTask(JsonFileTaskRepository(path), ConsoleNotifier())

### Drive it from the CLI

`cli` is the **driving adapter**. It parses the string `"done 1"` into the call `use_case.execute(1)`.
The `[console]` line comes from `ConsoleNotifier`, at the far end of the core.

In [4]:
cli("done 1", use_case)

[console] Task 'Write report' completed


### Read the result back

`JsonFileTaskRepository.save()` turned the `Task` back into a JSON dict (`asdict(task)`). The core never saw a dict.

In [5]:
print(path.read_text())

{
  "1": {
    "id": 1,
    "title": "Write report",
    "done": true
  }
}


## 2. Test wiring: in-memory + spy

This is the body of `demo_test()`. The core is the same code; only the adapters change. There is no disk, no stdout, and nothing to clean up.

### Wire fakes into the same ports

`InMemoryTaskRepository` fills the repository port with a dict. `SpyNotifier` fills the notifier port with a list that records each message.

In [6]:
repo = InMemoryTaskRepository([Task(1, "Write report")])
spy = SpyNotifier()

repo.get(1)

Task(id=1, title='Write report', done=False)

### A test is just another driver

There is no `cli` here. The test calls the driving port directly, so nothing prints: the notification went into the spy.

In [7]:
CompleteTask(repo, spy).execute(1)

### Assert on the spy, not on stdout

The test checks state through the port (`repo.get`) and messages through the spy. It never captures or parses console output.

In [8]:
assert repo.get(1).done is True
assert spy.sent == ["Task 'Write report' completed"]
print("assertions passed:", spy.sent)

assertions passed: ["Task 'Write report' completed"]


## 3. The rule lives in the core

Both tasks are now done. Complete each again, once through each wiring.
The same `ValueError` comes back both times: `Task.complete()` enforces the rule, whichever driver called it and whichever store holds the task.

In [9]:
for wiring, run in [
    ("production (cli + JSON)", lambda: cli("done 1", use_case)),
    ("test (direct + memory)",  lambda: CompleteTask(repo, spy).execute(1)),
]:
    try:
        run()
    except ValueError as e:
        print(f"{wiring:<24} → ValueError: {e}")

production (cli + JSON)  → ValueError: Task 1 is already done
test (direct + memory)   → ValueError: Task 1 is already done
